In [ ]:
!nvidia-smi
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
print("Dataset downloaded to:", path)

import os
os.listdir(path)

In [ ]:
import pandas as pd

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(df.shape)
df.head()

In [ ]:
import re

def normalize_symptom(value):
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s

label_col = df.columns[0]
SYMPTOM_COLS = [c for c in df.columns if c != label_col]

df[label_col] = df[label_col].astype(str).str.strip()

symptom_vocab = [normalize_symptom(c) for c in SYMPTOM_COLS]
df = df.rename(columns=dict(zip(SYMPTOM_COLS, symptom_vocab)))

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

X = df[symptom_vocab].to_numpy(dtype=np.float32)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[label_col])

In [ ]:
from sklearn.model_selection import train_test_split

MIN_SAMPLES_PER_CLASS = 5

class_counts = pd.Series(y).value_counts()
keep_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index
keep_mask = pd.Series(y).isin(keep_classes).to_numpy()

X_filtered = X[keep_mask]
y_filtered = y[keep_mask]

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)